## EDA in Python
This notebook documents steps taken to perform exploratory data analysis in Python, including loading data from MySQL database using SQLAlchemy.

In [2]:
# Import the important libraries
import sqlalchemy as sa
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymysql

In [3]:
# Create a connection to MySQL database using SQLAlchemy
# Replace with your actual database credentials
username = 'root'
password = 'Mysql123'
host = 'localhost'  # or your remote host IP/domain
port = 3306  # default MySQL port
database = 'obeta_db'

# Create the connection string
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'

# Create the engine
engine = create_engine(connection_string, pool_pre_ping=True)

# Test the connection
try:
    with engine.connect() as connection:
        print("✓ Successfully connected to MySQL database!")
except Exception as e:
    print(f"✗ Connection failed: {e}")

✓ Successfully connected to MySQL database!


In [16]:
# Efficient way to get random records using WHERE with random ID
# This uses the primary key instead of ORDER BY RAND()
query = """
SELECT * FROM pick_data 
WHERE id >= (SELECT FLOOR(RAND() * (SELECT MAX(id) FROM pick_data)))
LIMIT 10000000;
"""

# Load data into a pandas DataFrame
pick_data_df = pd.read_sql(query, engine)

# Display basic information about the dataset
print(f"\nShape of pick_data table: {pick_data_df.shape}")
print(f"\nFirst few rows:")
print(pick_data_df.head())
print(f"\nData types:")
print(pick_data_df.dtypes)
print(f"\nDataset info:")
print(pick_data_df.info())


Shape of pick_data table: (10000000, 9)

First few rows:
      id product_id warehouse_section  origin order_number position_in_order  \
0  10158     100608        Kabellager      48     06487012              0001   
1  10496     100608        Kabellager      48     06925577              0002   
2  12033     100608        Kabellager      48     05324394              0002   
3  15250     100608        Kabellager      48     02686252              0001   
4  17255     100609        Kabellager      48     05869731              0007   

   pick_volume quantity_unit        date  
0          100            Mt  2017-03-13  
1          100            Mt  2017-06-06  
2          200            Mt  2018-04-26  
3          100            Mt  2020-03-17  
4          500            Mt  2018-08-07  

Data types:
id                    int64
product_id           object
warehouse_section    object
origin                int64
order_number         object
position_in_order    object
pick_volume           

In [18]:
# Find the product ID where pick_volume equals its minimum value
min_pick_volume = pick_data_df['pick_volume'].min()
product_id_min_volume = pick_data_df[pick_data_df['pick_volume'] == min_pick_volume]['product_id'].values

print(f"Minimum pick_volume: {min_pick_volume}")
print(f"Product ID(s) with minimum pick_volume: {product_id_min_volume}")

Minimum pick_volume: 0
Product ID(s) with minimum pick_volume: ['101683' '107101' '110762' ... '112767' '205850' 'T28452']


In [17]:
print(pick_data_df.describe())

                 id        origin   pick_volume
count  1.000000e+07  1.000000e+07  1.000000e+07
mean   1.735297e+07  4.754090e+01  5.247455e+01
std    6.135557e+06  8.410843e-01  3.399107e+02
min    1.015800e+04  4.600000e+01  0.000000e+00
25%    1.301323e+07  4.800000e+01  1.000000e+00
50%    1.840594e+07  4.800000e+01  4.000000e+00
75%    2.254290e+07  4.800000e+01  2.000000e+01
max    2.603146e+07  4.800000e+01  2.000000e+05


In [4]:
pd_neg_query = """
SELECT * FROM pick_data
WHERE pick_volume < 0;
"""
pd_negative_volume = pd.read_sql(pd_neg_query, engine)
print(pd_negative_volume.describe())
print(pd_negative_volume.head())

                 id      origin  pick_volume
count  1.000000e+02  100.000000   100.000000
mean   3.101992e+07   47.860000   -34.900000
std    1.143188e+06    0.512865   201.359044
min    2.857336e+07   46.000000 -2000.000000
25%    3.023114e+07   48.000000   -10.000000
50%    3.161643e+07   48.000000    -3.000000
75%    3.186919e+07   48.000000    -1.000000
max    3.193038e+07   48.000000    -1.000000
         id product_id warehouse_section  origin order_number  \
0  28573365     R12006               SHL      48     01670127   
1  28573803     294723               SHL      48     01672819   
2  28664763     234142               SHL      48     01723719   
3  28675340     206454               SHL      48     01728544   
4  28687988     R20556               SHL      48     01737252   

  position_in_order  pick_volume quantity_unit        date  
0              0004           -3            St  2015-09-04  
1              0006           -1            St  2015-09-04  
2              0009  